# AI Interview Coach — Pipeline Demo

This notebook demonstrates the full NLP pipeline for the AI Interview Coach project, running each stage end-to-end: text metrics, filler detection, structure analysis, relevance scoring, and the combined pipeline analyzer.

In [ ]:
import sys
sys.path.insert(0, '..')

from backend.nlp.pipeline import InterviewAnalyzer
from backend.nlp.text_metrics import compute_all_metrics
from backend.nlp.filler_detection import detect_fillers
from backend.nlp.structure_analysis import analyze_structure
from backend.nlp.relevance_scoring import compute_relevance
import pandas as pd
import json

## Step 1: Sample Interview Q&A

In [ ]:
question = "Та өөрийгөө танилцуулна уу."
response = "Намайг Бат гэдэг. Би МУИС-ийн Мэдээллийн технологийн сургуулийг төгссөн. Төгссөнөөсөө хойш 3 жил программ хангамжийн инженерээр ажилласан. Тэгээд өмнөх компанидаа багийн ахлагчаар хариуцсан. Би шинэ технологи судлах дуртай бөгөөд багаар хамтран ажиллах чадвартай."
print(f"Асуулт: {question}")
print(f"Хариулт: {response}")

## Step 2: Text Metrics

In [ ]:
metrics = compute_all_metrics(response)
for key, val in metrics.items():
    print(f"  {key}: {val}")

## Step 3: Filler Word Detection

In [ ]:
fillers = detect_fillers(response)
print(f"Total fillers: {fillers['total_count']}")
print(f"Mongolian: {fillers['mongolian']}")
print(f"English: {fillers['english']}")

## Step 4: Structure Analysis

In [ ]:
structure = analyze_structure(response)
print(f"Action verbs found: {structure['action_verbs']['count']}")
for v in structure['action_verbs']['verbs']:
    print(f"  - {v['verb']} ({v['language']})")
print(f"\nSTAR score: {structure['star_method']['score']}")
for element, data in structure['star_method']['elements'].items():
    status = "✓" if data['detected'] else "✗"
    print(f"  {status} {element}")

## Step 5: Relevance Scoring

In [ ]:
job_desc = "Программ хангамжийн инженер. Веб болон мобайл аппликейшн хөгжүүлэх. Баг удирдах."
required_skills = "Багаар ажиллах, Харилцааны чадвар, Шинжилгээ хийх"
relevance = compute_relevance(response, job_desc, required_skills)
print(f"TF-IDF score: {relevance['tfidf_score']}")
print(f"Keyword match score: {relevance['keyword_match']['score']}")
print(f"Combined: {relevance['combined_score']}")
print(f"Matched skills: {relevance['keyword_match']['matched_skills']}")
print(f"Missing skills: {relevance['keyword_match']['missing_skills']}")

## Step 6: Full Pipeline Analysis

In [ ]:
analyzer = InterviewAnalyzer()
result = analyzer.analyze_response(
    response_text=response,
    question=question,
    job_description=job_desc,
    required_skills=required_skills,
)
print(json.dumps(result['feedback'], indent=2, ensure_ascii=False))

## Step 7: Batch Analysis on Dataset

In [ ]:
import glob, os
mn_dir = '../backend/data/processed/mn/'
frames = []
for f in sorted(glob.glob(os.path.join(mn_dir, '*.csv'))):
    tmp = pd.read_csv(f)
    if 'company' not in tmp.columns:
        tmp.insert(0, 'company', 'Unitel Group')
    frames.append(tmp)
df = pd.concat(frames, ignore_index=True)
print(f"Loaded {len(df)} Q&A rows")

# Analyze 5 random samples
samples = df.sample(5, random_state=42)
for _, row in samples.iterrows():
    res = analyzer.analyze_response(row['хариулт'], row['асуулт'])
    print(f"\nCompany: {row['company']}")
    print(f"Q: {row['асуулт'][:60]}...")
    print(f"Score: {res['feedback']['overall_score']}")
    print(f"Words: {res['metrics']['text']['word_count']}, Fillers: {res['metrics']['fillers']['total_count']}")

## Evaluation Metrics

In [ ]:
scores = []
for _, row in df.iterrows():
    res = analyzer.analyze_response(row['хариулт'], row['асуулт'])
    scores.append(res['feedback']['overall_score'])
df['score'] = scores
print(f"Score statistics:")
print(df['score'].describe().round(2))

## Limitations
- Mongolian tokenization is whitespace-based (no morphological analysis)
- STAR detection uses simple keyword matching
- Relevance scoring works best when job listing has structured required_skills
- Small dataset (471 Q&A pairs) limits statistical power